In [1]:
import pandas as pd
# Adjust settings to display all rows and columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 1000)  # Set a wide display width to fit the content
pd.set_option('display.max_colwidth', None)  # Show full column content without truncation


# RUN 1

In [2]:
#df.to_pickle("24_df.pkl")

In [ ]:
direc_jpg = 'images/'
direc_tables = 'tables/'
my_aiff  ='/Volumes/MUSIC_PROD/_1_NEW_SOURCE copy/_2024_this'
#my_aiff ="/Volumes/MUSIC_PROD/_1_NEW_SOURCE copy"#/_2025_this"#/test"
#my_aiff ="/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/cover"
exec(open("run.py",encoding="utf-8").read())
# Save the selected DataFrame portion to a pickle file
#df.to_pickle("25_df.pkl")
df.head(1)

<string>:209: DeprecationWarning: 'aifc' is deprecated and slated for removal in Python 3.13


   Sample Rate (Hz)  Bit Depth  Count
0             44100         16    725

ALL AIFF in correct format -> if something changes go erase the already downsized
 


ENTER to analyze 
 


Searching for Files: 100%|████████████████████████████████████████████| 729/729 [00:00<00:00, 77254.29files/s]


ATTN ::: <E>  AFTER making sure that you have assigned the DIRECTORY PATH to ::: var ::: /Volumes/MUSIC_PROD/_1_NEW_SOURCE copy/_2024_this
***df*** will be returned having ::: 725  rows, write df in next cell to see your DATA FRAME

second part FREQUENCIES OF df :::

Grand Total Size in MB: 44266.350
Grand Total Size in Bytes: 46416631986.000
Grand Total Size in GB: 43.229

 FREQUENCY TABLE::: 

  Extension  frequency  total_size_in_mb
0     .aiff        725      44266.349779


Computing LUFS:  21%|███████████                                          | 151/725 [01:49<07:42,  1.24file/s]

# bar plot freq distribution 

In [ ]:
# -----######-----######-----######-----######-----######-----#
# CORE FUNCTION: _freq_1604_i1_GET_tables_dynamic_range_stats
# -----######-----######-----######-----######-----######-----#

import os
import numpy as np
import pandas as pd
import librosa
from scipy.signal import butter, filtfilt
from tqdm import tqdm

def _freq_1604_i1_GET_tables_dynamic_range_stats(
    df,
    output_dir,
    path_col='Path',
    id_col='ID'
):
    """
    For each audio file in df, computes dynamic range stats across 9 frequency bands (freq & time domain),
    and saves results as `table_freq_{ID}.csv` in the specified directory.

    Parameters:
        df (pd.DataFrame): Must include audio file paths and IDs.
        output_dir (str): Destination to save CSVs.
        path_col (str): Column name for audio paths.
        id_col (str): Column name for unique IDs.
    """

    os.makedirs(output_dir, exist_ok=True)

    bands = {
        "Low-Low": (20, 60),
        "Low-Mid": (60, 120),
        "Low-High": (120, 200),
        "Mid-Low": (200, 500),
        "Mid-Mid": (500, 1000),
        "Mid-High": (1000, 2000),
        "High-Low": (2000, 5000),
        "High-Mid": (5000, 10000),
        "High-High": (10000, 22050)  # fallback limit, to be updated per file
    }

    def bandpass_filter(y, sr, lowcut, highcut):
        nyquist = 0.5 * sr
        low = lowcut / nyquist
        high = highcut / nyquist
        if low <= 0 or high >= 1:
            return np.zeros_like(y)  # return silence if invalid
        b, a = butter(2, [low, high], btype="band")
        return filtfilt(b, a, y)

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="TQM Processing"):
        path = row[path_col]
        ID = row[id_col]
        try:
            y, sr = librosa.load(path, sr=None)
            bands['High-High'] = (10000, sr * 0.49)
            S = np.abs(librosa.stft(y, n_fft=2048, hop_length=1024))
            S_db = librosa.amplitude_to_db(S, ref=np.max)
            freqs = librosa.fft_frequencies(sr=sr, n_fft=S.shape[0])

            mean_dbs, max_dbs, min_dbs = {}, {}, {}
            rms_values = {}
            freq_domain_dynamics = {}
            time_domain_dynamics = {}

            for band_name, (low, high) in bands.items():
                band_idx = np.where((freqs >= low) & (freqs <= high))[0]
                band_dbs = S_db[band_idx, :]
                band_amps = S[band_idx, :]

                mean_dbs[band_name] = np.mean(band_dbs)
                max_dbs[band_name] = np.max(band_dbs)
                min_dbs[band_name] = np.min(band_dbs)

                rms = np.sqrt(np.mean(band_amps**2))
                peak = np.max(band_amps)
                freq_domain_dynamics[band_name] = 20 * np.log10(peak / rms) if rms > 0 else 0
                rms_values[band_name] = rms

                y_filt = bandpass_filter(y, sr, low, high)
                rms_t = np.sqrt(np.mean(y_filt**2))
                peak_t = np.max(np.abs(y_filt))
                time_domain_dynamics[band_name] = 20 * np.log10(peak_t / rms_t) if rms_t > 0 else 0

            overall_mean = np.mean(S_db)
            overall_max = np.max(S_db)
            overall_min = np.min(S_db)
            overall_rms = np.sqrt(np.mean(S**2))
            overall_peak = np.max(S)
            freq_dr_overall = 20 * np.log10(overall_peak / overall_rms) if overall_rms > 0 else 0

            rms_y = np.sqrt(np.mean(y**2))
            peak_y = np.max(np.abs(y))
            time_dr_overall = 20 * np.log10(peak_y / rms_y) if rms_y > 0 else 0

            df_out = pd.DataFrame({
                'Band': list(bands.keys()) + ['Overall Song'],
                'Mean dB': list(mean_dbs.values()) + [overall_mean],
                'Max dB': list(max_dbs.values()) + [overall_max],
                'Min dB': list(min_dbs.values()) + [overall_min],
                'RMS': list(rms_values.values()) + [overall_rms],
                'Frequency-Domain_DR': list(freq_domain_dynamics.values()) + [freq_dr_overall],
                'Time-Domain_DR': list(time_domain_dynamics.values()) + [time_dr_overall]
            })

            df_out.to_csv(os.path.join(output_dir, f"table_freq_{ID}.csv"), index=False)
        except Exception as e:
            print(f"❌ ERROR on {ID} | {path} ::: {e}")



In [ ]:
# Example: compute and save to folder
_freq_1604_i1_GET_tables_dynamic_range_stats(
    df,
    output_dir=direc_tables
)
df['Path_csv_freq'] = f'{direc_tables}' + 'table_freq_' + df['ID'] + '.csv'

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# import os

# def plot_dbs_heatmap_split_bars(df, save_path=None):
#     """
#     Plots a heatmap of dB levels with split bars for individual bands and the overall song.
#     Optionally saves the plot if `save_path` is provided.
#     Returns the figure object.
#     """
#     bands = df['Band'][:-1].tolist()
#     overall_stats = df.iloc[-1]
    
#     colors = [
#         'blue', 'cyan', 'lightblue', 'green', 'lightgreen', 
#         'lime', 'red', 'orange', 'yellow'
#     ]

#     fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(5, 8), gridspec_kw={'height_ratios': [2, 1], 'hspace': 0.15})
#     bar_width = 0.98

#     # ----- Upper Plot -----
#     for i, (_, row) in enumerate(df[:-1].iterrows()):
#         min_dB = row['Min dB']
#         max_dB = row['Max dB']
#         mean_dB = row['Mean dB']

#         ax1.fill_betweenx([0, 1], i - bar_width / 2, i + bar_width / 2, color=colors[i % len(colors)], edgecolor='black', linewidth=0.8)
#         ax1.hlines((mean_dB + 100) / 110, i - bar_width / 2, i + bar_width / 2, color='black', linewidth=1.2)
#         ax1.scatter(i, (min_dB + 100) / 110, color='white', edgecolor='black', zorder=5)
#         ax1.scatter(i, (max_dB + 100) / 110, color='black', zorder=5)
#         ax1.scatter(i, (mean_dB + 100) / 110, color='blue', zorder=5)

#     overall_index = len(bands)
#     ax1.fill_betweenx([0, 1], overall_index - bar_width / 2, overall_index + bar_width / 2, color='#00796B', edgecolor='black', linewidth=0.8)
#     ax1.hlines((overall_stats['Mean dB'] + 100) / 110, overall_index - bar_width / 2, overall_index + bar_width / 2, color='black', linewidth=2)
#     ax1.hlines((overall_stats['Min dB'] + 100) / 110, overall_index - bar_width / 2, overall_index + bar_width / 2, color='black', linestyle='--', linewidth=2)
#     ax1.hlines((overall_stats['Max dB'] + 100) / 110, overall_index - bar_width / 2, overall_index + bar_width / 2, color='black', linestyle='-.', linewidth=2)

#     ax1.set_xticks(range(len(bands) + 1))
#     ax1.set_xticklabels([''] * (len(bands) + 1))
#     ax1.set_xlim(-0.5, len(bands) + 0.5)
#     ax1.set_ylim(0, 1)
#     ax1.set_yticks(np.linspace(0, 1, 12))
#     ax1.set_yticklabels(np.linspace(-100, 10, 12).astype(int))
#     ax1.set_title("Frequency Band dB Levels", fontsize=14)
#     for spine in ['top', 'right', 'left']:
#         ax1.spines[spine].set_visible(False)

#     # ----- Lower Plot -----
#     offset = 0.2
#     ax2.bar(np.arange(len(bands)) - offset, df['Time-Domain_DR'][:-1], color='skyblue', edgecolor='black', width=bar_width / 2, label='Time-Domain')
#     ax2.bar(len(bands) - offset, df['Time-Domain_DR'].iloc[-1], color='#00796B', edgecolor='black', width=bar_width / 2)
#     ax2.bar(np.arange(len(bands)) + offset, df['Frequency-Domain_DR'][:-1], color='lightgreen', edgecolor='black', width=bar_width / 2, label='Frequency-Domain')
#     ax2.bar(len(bands) + offset, df['Frequency-Domain_DR'].iloc[-1], color='#00796B', edgecolor='black', width=bar_width / 2)

#     ax2.set_xticks(range(len(bands) + 1))
#     ax2.set_xticklabels([''] * (len(bands) + 1))
#     ax2.set_xlim(-0.5, len(bands) + 0.5)
#     ax2.set_ylim(0, 40)
#     ax2.set_yticks(np.linspace(0, 30, 6))
#     ax2.set_ylabel('Dynamic Range (dB)', fontsize=10)
#     ax2.set_title("Time-Domain and Frequency-Domain DR", fontsize=12)

#     ax2_rms = ax2.twinx()
#     ax2_rms.set_ylim(0, 100)
#     ax2_rms.set_ylabel('RMS', fontsize=10, color='darkorange')
#     ax2_rms.tick_params(axis='y', labelcolor='darkorange', direction='out', pad=5)
#     ax2_rms.plot(np.arange(len(bands)), df['RMS'][:-1], color='darkorange', linewidth=2, marker='o', label='RMS')
#     ax2_rms.plot(len(bands), df['RMS'].iloc[-1], color='darkorange', linewidth=2, marker='o')

#     ax2.legend(loc='upper left')
#     ax2_rms.legend(loc='upper right')

#     for i, color in enumerate(colors):
#         ax2.plot(i, -2, marker='o', markersize=10, color=color, clip_on=False)
#     ax2.plot(len(bands), -2, marker='o', markersize=10, color='#00796B', clip_on=False)

#     fig.subplots_adjust(top=0.95, bottom=0.05, left=0.1, right=0.9, hspace=0.25)

#     # ----- Save If Needed -----
#     if save_path:
#         os.makedirs(os.path.dirname(save_path), exist_ok=True)
#         fig.savefig(save_path, bbox_inches='tight', dpi=300)
#         print(f"TQM: Plot saved to {save_path}")
#     else:
#         print("TQM: Plot created (not saved).")

#     return fig

# # -----######-----######-----######-----######-----######-----#
# # 0_FNS: _plot_1604_batch_GET_save_png_dbs
# # -----######-----######-----######-----######-----######-----#
# import os
# import pandas as pd

# def _plot_1604_batch_GET_save_png_dbs(df, output_dir):
#     """
#     For each row in df, loads the 'Path_csv_freq' as a DataFrame, 
#     generates the dB/DR heatmap, and saves it as PNG to output_dir.

#     Parameters:
#         df (pd.DataFrame): Must include 'Path_csv_freq' and 'ID'
#         output_dir (str): Folder where plots will be saved
#     """
#     from matplotlib import pyplot as plt

#     os.makedirs(output_dir, exist_ok=True)

#     for idx, row in df.iterrows():
#         path_csv = row['Path_csv_freq']
#         track_id = row['ID']
#         save_path = os.path.join(output_dir, f'dbs_plot_{track_id}.png')

#         try:
#             df_freq = pd.read_csv(path_csv)
#             fig = plot_dbs_heatmap_split_bars(df_freq, save_path=save_path)
#             plt.close(fig)
#             print(f"✅ Saved: {save_path}")
#         except Exception as e:
#             print(f"❌ Error processing {path_csv}: {e}")


In [ ]:
_plot_1604_batch_GET_save_png_dbs(df, output_dir=direc_jpg)
df['Path_png_dr'] = f'{direc_jpg}' + 'dbs_plot_' + df['ID'] + '.png'

# embed 

In [ ]:
# # -----######-----######-----######-----######-----######-----######-----
# # CORE FUNCTION: _dr_1604_i2_GET_embedded_album_dr_custom
# # -----######-----######-----######-----######-----######-----######-----
# import os
# from PIL import Image
# import pandas as pd

# def _dr_1604_i2_GET_embedded_album_dr_custom(
#     df,
#     flip_deg=180,           # Rotate 180° by default
#     scale=1.0,              # 1.0 = original size
#     x_offset=0,             # Fine-tune position (horizontal)
#     y_offset=0,             # Fine-tune position (vertical)
#     position="center",      # Now supports: center + top_left/top_right/bottom_left/bottom_right/custom
#     custom_coords=None      # Only used if position == "custom"
# ):
#     """
#     Embeds each DR PNG (flipped, scaled, centered) over its matching album cover.

#     Parameters:
#         df (pd.DataFrame): Requires 'Path_png_dr', 'Path_jpg_album'
#         flip_deg (int): Degrees to rotate DR overlay
#         scale (float): Scale factor for DR overlay
#         x_offset, y_offset (int): Positional fine-tuning
#         position (str): One of ['center', 'top_left', 'top_right', 'bottom_left', 'bottom_right', 'custom']
#         custom_coords (tuple): Exact (x, y) if position == "custom"
#     """
#     for idx, row in df.iterrows():
#         try:
#             # Load both images
#             base_img = Image.open(row['Path_jpg_album']).convert("RGBA")
#             overlay = Image.open(row['Path_png_dr']).convert("RGBA")

#             # Rotate and scale
#             overlay = overlay.rotate(flip_deg, expand=True)
#             new_size = (int(overlay.width * scale), int(overlay.height * scale))
#             overlay = overlay.resize(new_size, Image.ANTIALIAS)

#             # Get base and overlay sizes
#             bx, by = base_img.size
#             ox, oy = overlay.size

#             # Calculate position
#             if position == "custom" and custom_coords:
#                 pos = custom_coords
#             elif position == "center":
#                 pos = ((bx - ox) // 2 + x_offset, (by - oy) // 2 + y_offset)
#             else:
#                 pos_dict = {
#                     "top_left": (x_offset, y_offset),
#                     "top_right": (bx - ox - x_offset, y_offset),
#                     "bottom_left": (x_offset, by - oy - y_offset),
#                     "bottom_right": (bx - ox - x_offset, by - oy - y_offset)
#                 }
#                 pos = pos_dict.get(position, (x_offset, y_offset))

#             # Composite and save
#             base_img.paste(overlay, pos, overlay)
#             base_img.convert("RGB").save(row['Path_jpg_album'])

#         except Exception as e:
#             print(f"❌ Error embedding DR PNG for row {idx}: {e}")


In [ ]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

_dr_1604_i2_GET_embedded_album_dr_custom(
    df,
    flip_deg=180 + 90,
    scale=.282,
    x_offset=-169,
    y_offset=93,
    position="center"
)


# Music note written  and # ID written 

In [ ]:
# # -----######-----######-----######-----######-----######-----#
# # 0_FNS: _keyid_1604_i3_GET_png_label_blocks_rightflush
# # -----######-----######-----######-----######-----######-----#

# import os
# import pandas as pd
# import matplotlib.pyplot as plt

# def _keyid_1604_i3_GET_png_label_blocks_rightflush(df, save_dir='img_keyid'):
#     """
#     For each row (ID, KEY), create a 2-block image:
#       - TOP block: white bg, right-aligned KEY inside block
#       - BOTTOM block: washed white, right-aligned ID inside block

#     Exports as: key_and_id_{ID}.png into save_dir
#     """
#     os.makedirs(save_dir, exist_ok=True)

#     for _, row in df.iterrows():
#         ID = row['ID']
#         KEY = row['KEY']
#         fname = f"key_and_id_{ID}.png"
#         out_path = os.path.join(save_dir, fname)

#         fig, ax = plt.subplots(figsize=(4, 4))
#         ax.set_xlim(0, 1)
#         ax.set_ylim(0, 1)
#         ax.axis('off')

#         # Background blocks
#         ax.fill_between([0, 1], 0.5, 1, color='white')      # Top half
#         ax.fill_between([0, 1], 0.0, 0.5, color='#f2f2f2')   # Bottom half

#         # Texts - aligned inside their rectangles (flush right, no margin)
#         ax.text(
#             0.98, 0.75, str(KEY),
#             fontsize=58, fontweight='bold',
#             ha='right', va='center', color='black'
#         )
#         ax.text(
#             0.98, 0.25, str(ID),
#             fontsize=31, fontweight='bold',
#             ha='right', va='center', color='black'
#         )

#         plt.savefig(out_path, bbox_inches='tight', dpi=150)
#         plt.close()


In [ ]:
_keyid_1604_i3_GET_png_label_blocks_rightflush(df, save_dir=direc_jpg)
df['Path_png_id_and_key'] = f'{direc_jpg}' + 'key_and_id_' + df['ID'] + '.png'

# embed 

In [ ]:
# # -----######-----######-----######-----######-----######-----######-----
# # CORE FUNCTION: _idkey_1604_i1_GET_embedded_album_idkey_right
# # -----######-----######-----######-----######-----######-----######-----
# import os
# from PIL import Image
# import pandas as pd

# def _idkey_1604_i1_GET_embedded_album_idkey_right(
#     df,
#     flip_deg=0,             # Rotate ID+KEY PNG if needed
#     scale=1.0,              # Scale factor
#     x_offset=0,             # How far from the right edge
#     y_offset=0              # Fine-tune vertical alignment
# ):
#     """
#     Embeds ID+KEY PNG on the right center of each album JPG, optionally flipped and scaled.

#     Parameters:
#         df (pd.DataFrame): Requires 'Path_png_id_and_key', 'Path_jpg_album'
#         flip_deg (int): Degrees to rotate the overlay
#         scale (float): Scale factor for overlay image
#         x_offset (int): Shift left from right edge
#         y_offset (int): Shift up/down from vertical center
#     """
#     for idx, row in df.iterrows():
#         try:
#             base_img = Image.open(row['Path_jpg_album']).convert("RGBA")
#             overlay = Image.open(row['Path_png_id_and_key']).convert("RGBA")

#             # Flip/rotate + scale
#             overlay = overlay.rotate(flip_deg, expand=True)
#             new_size = (int(overlay.width * scale), int(overlay.height * scale))
#             overlay = overlay.resize(new_size, Image.ANTIALIAS)

#             bx, by = base_img.size
#             ox, oy = overlay.size

#             # Position: Centered vertically, right aligned
#             pos_x = bx - ox - x_offset
#             pos_y = (by - oy) // 2 + y_offset
#             position = (pos_x, pos_y)

#             base_img.paste(overlay, position, overlay)
#             base_img.convert("RGB").save(row['Path_jpg_album'])

#         except Exception as e:
#             print(f"❌ Error embedding ID+KEY PNG for row {idx}: {e}")


In [ ]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

_idkey_1604_i1_GET_embedded_album_idkey_right(
    df,
    flip_deg=0,
    scale=.68,
    x_offset=1,   # small padding from right
    y_offset=80     # keep vertically centered
)


# embed 

In [ ]:
_aiff_2409_i1_GET_embed_aiff_covers(df)

# COMMENT 

# RENAME

# save master df -> only one 